# SportsGraph — Phase 2: YOLO Fine-Tuning (Soccer 4-Class)

Self-contained Kaggle runner for **sports-graph** fine-tuning on GPU.

### Objectives & Architecture
- **4 Classes**: `0: ball`, `1: goalkeeper`, `2: player`, `3: referee`
- **High Resolution**: Trained at `imgsz=1280` so soccer balls remain visible
- **Regularization Augmentations**: Mosaic (1.0), MixUp (0.15), HSV jitter, flips
- **Deterministic Time Split**: 80% train, 20% held-out validation
- **Target Benchmark**: Exceed baseline (**0.1767 mAP**) targeting **>0.85 mAP**
- **Self-Contained**: No external git clone required

### 1. Install Ultralytics & Import Dependencies

In [ ]:
!pip install --upgrade ultralytics pyyaml pillow pandas

import shutil
import zipfile
from pathlib import Path

import yaml

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data" / "warmup_labeled"
MODELS_DIR = WORKING_DIR / "models" / "fine_tuned"

print("Environment initialized.")

### 2. Dataset Setup & Helpers (Self-Contained)
Unpacks the Roboflow 4-class dataset from `/kaggle/input`.

In [ ]:
def unpack_roboflow_zip(zip_path: Path, target_dir: Path) -> None:
    target_images = target_dir / "images"
    target_labels = target_dir / "labels"
    target_images.mkdir(parents=True, exist_ok=True)
    target_labels.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        if "data.yaml" in z.namelist():
            data_yaml = yaml.safe_load(z.read("data.yaml").decode("utf-8"))
            class_names = data_yaml.get("names", [])
        else:
            class_names = ["ball", "goalkeeper", "player", "referee"]

        classes_file = target_dir / "classes.txt"
        classes_file.write_text("\n".join(class_names) + "\n", encoding="utf-8")
        print(f"Classes: {class_names}")

        extracted_images = 0
        extracted_labels = 0
        for member in z.namelist():
            for s in ["train", "valid", "test"]:
                if member.startswith(f"{s}/images/") and member.lower().endswith(
                    (".jpg", ".png", ".jpeg")
                ):
                    filename = Path(member).name
                    dest_img = target_images / filename
                    dest_img.write_bytes(z.read(member))
                    extracted_images += 1

                    label_member = f"{s}/labels/{Path(member).stem}.txt"
                    dest_label = target_labels / f"{Path(member).stem}.txt"
                    if label_member in z.namelist():
                        dest_label.write_bytes(z.read(label_member))
                    else:
                        dest_label.write_text("")
                    extracted_labels += 1

        print(f"Extracted {extracted_images} images and {extracted_labels} labels.")


def unpack_from_unzipped_folder(input_dir: Path, target_dir: Path) -> None:
    target_images = target_dir / "images"
    target_labels = target_dir / "labels"
    target_images.mkdir(parents=True, exist_ok=True)
    target_labels.mkdir(parents=True, exist_ok=True)

    data_yaml_paths = list(input_dir.rglob("data.yaml"))
    if data_yaml_paths:
        with open(data_yaml_paths[0]) as f:
            data_yaml = yaml.safe_load(f)
        class_names = data_yaml.get("names", ["ball", "goalkeeper", "player", "referee"])
    else:
        class_names = ["ball", "goalkeeper", "player", "referee"]

    (target_dir / "classes.txt").write_text("\n".join(class_names) + "\n")
    print(f"Classes: {class_names}")

    extracted_images = 0
    for img_path in input_dir.rglob("*.jpg"):
        dest_img = target_images / img_path.name
        if not dest_img.exists():
            shutil.copy2(img_path, dest_img)
            extracted_images += 1
        label_cand = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
        dest_label = target_labels / f"{img_path.stem}.txt"
        if label_cand.exists():
            shutil.copy2(label_cand, dest_label)
        elif not dest_label.exists():
            dest_label.write_text("")

    print(f"Gathered {extracted_images} images from {input_dir}")


# Locate dataset in /kaggle/input
input_path = Path("/kaggle/input")
zip_files = list(input_path.rglob("*.zip")) if input_path.exists() else []

if zip_files:
    print(f"Found zip dataset: {zip_files[0]}")
    unpack_roboflow_zip(zip_files[0], DATA_DIR)
elif input_path.exists() and any(input_path.rglob("*.jpg")):
    print(f"Found unzipped dataset in: {input_path}")
    unpack_from_unzipped_folder(input_path, DATA_DIR)
else:
    raise FileNotFoundError(
        "No dataset found in /kaggle/input! Please click '+ Add Input' on the right, "
        "upload the 18.8MB 'football-players-detection...zip', and re-run."
    )

### 3. Time-Based Split & Manifest Generation
Applies the exact 80/20 time-based split matching our local evaluation benchmark.

In [ ]:
def prepare_training_manifest(dataset_dir: Path, held_out_frac: float = 0.2) -> Path:
    images_dir = dataset_dir / "images"
    classes_file = dataset_dir / "classes.txt"
    class_names = [line.strip() for line in classes_file.read_text().splitlines() if line.strip()]
    image_paths = sorted(images_dir.glob("*.jpg")) + sorted(images_dir.glob("*.png"))

    # Time-based split: chronological order by filename
    ordered = sorted(image_paths, key=lambda p: p.name)
    split_idx = int(len(ordered) * (1 - held_out_frac))
    split_idx = max(1, min(split_idx, len(ordered) - 1))
    working_paths, held_out_paths = ordered[:split_idx], ordered[split_idx:]

    train_txt = dataset_dir / "train.txt"
    val_txt = dataset_dir / "val.txt"
    # Write absolute resolved paths so Ultralytics finds them from any cwd
    train_txt.write_text("\n".join(str(p.resolve().as_posix()) for p in working_paths) + "\n")
    val_txt.write_text("\n".join(str(p.resolve().as_posix()) for p in held_out_paths) + "\n")

    manifest = {
        "path": str(dataset_dir.as_posix()),
        "train": str(train_txt.resolve().as_posix()),
        "val": str(val_txt.resolve().as_posix()),
        "nc": len(class_names),
        "names": class_names,
    }
    manifest_path = dataset_dir / "dataset.yaml"
    with open(manifest_path, "w") as f:
        yaml.safe_dump(manifest, f, sort_keys=False)

    print(
        f"Manifest prepared: {len(working_paths)} train, "
        f"{len(held_out_paths)} val -> {manifest_path}"
    )
    return manifest_path


manifest_file = prepare_training_manifest(DATA_DIR, held_out_frac=0.2)

### 4. Train Model on GPU with Transfer Learning & Regularization

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

train_args = {
    "data": str(manifest_file.resolve().as_posix()),
    "epochs": 50,
    "imgsz": 1280,
    "batch": 8,
    "device": "0",
    "project": str(MODELS_DIR.parent.as_posix()),
    "name": MODELS_DIR.name,
    "exist_ok": True,
    # Regularizing augmentations
    "mosaic": 1.0,
    "mixup": 0.15,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "fliplr": 0.5,
    "verbose": True,
}

print("Starting YOLOv8n fine-tuning on GPU...")
results = model.train(**train_args)

### 5. Verify & Download Trained Weights

In [ ]:
best_weights = MODELS_DIR / "weights" / "best.pt"
if best_weights.exists():
    size_mb = best_weights.stat().st_size / (1024 * 1024)
    print("Training completed successfully!")
    print(f"Best weights saved at: {best_weights} ({size_mb:.2f} MB)")
    print("\n--- Next Step ---")
    print("1. In the right-hand panel under 'Output', download best.pt")
    print("2. Place it in your local repository at: models/fine_tuned/weights/best.pt")
    cmd = (
        "uv run python -m sportsgraph.detection.run_baseline_eval "
        "--model models/fine_tuned/weights/best.pt"
    )
    print(f"3. Run local evaluation to verify against baseline:\n   {cmd}")
else:
    print("Weights file not found at expected location. Check training logs above.")